# 05. SARpyx to Rustamps burst PSI end-to-end

This notebook turns the burst-native SNAPflow pair demo into a PSI-oriented stack workflow. It searches for a same-burst Sentinel-1 stack, downloads the burst products, processes each master-secondary pair with SARpyx/SNAP, exports a StaMPS-compatible dataset, prepares `PATCH_*` inputs with StaMPS, and runs Rustamps stages 1-8.

The default area is the Rome AOI used by the source SNAPflow notebook. The default stack is intentionally constrained to one Sentinel-1A burst group so all acquisitions share `BurstId`, swath, relative orbit, orbit direction, and polarization.


In [ ]:
import importlib.util, os, shutil
needs = {
    "ESA SNAP `gpt` (Java)": shutil.which("gpt") is not None,
    "CDSE credentials (CDSE_USERNAME / CDSE_PASSWORD)": bool(os.getenv("CDSE_USERNAME") or os.getenv("CDSE_USR")) and bool(os.getenv("CDSE_PASSWORD") or os.getenv("CDSE_PSW")),
    "python package `phidown`": importlib.util.find_spec("phidown") is not None,
    "python package `sarpyx`": importlib.util.find_spec("sarpyx") is not None,
}
missing = [name for name, ok in needs.items() if not ok]
if missing:
    raise RuntimeError(
        "NOT PROVEN - notebook 05 is the upstream SNAP/CDSE preprocessing workflow, which is outside the Rust-only runtime. "
        "Missing: " + "; ".join(missing) + ". It downloads Sentinel-1 bursts and runs Java SNAP; Rustamps itself only consumes its output."
    )

## Workflow contract

External contracts used here:

- StaMPS is the persistent-scatterer / small-baseline time-series processor: https://github.com/dbekaert/StaMPS
- SNAP `StampsExport` expects a coregistered SLC stack plus interferograms with elevation: https://step.esa.int/main/wp-content/help/versions/13.0.0/snap-toolboxes/eu.esa.microwavetbx.sar.op.insar.ui/operators/StampsExportOp.html
- SNAP2StaMPS runs SNAP TOPSAR processing and exports inputs for StaMPS: https://github.com/mdelgadoblasco/snap2stamps

Rustamps starts after the StaMPS prep step has created the patch-level files such as `PATCH_1/pscands.1.ij`, `PATCH_1/pscands.1.ph`, and `PATCH_1/pscands.1.ll`.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import shutil
import subprocess
import sys
import time
from datetime import timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
np.set_printoptions(suppress=True, precision=4)

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
REPO_ROOT = NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebooks import utils as nb_utils
for _helper_name in ('load_cdse_env_file', 'first_env', 'display_path', 'ensure_runtime_dirs', 'prepend_path', 'ensure_gawk_on_path', 'configure_stamps_environment', 'content_start', 'yyyymmdd', 'resolve_executable', 'resolve_mt_prep_snap', 'gpt_looks_like_snap', '_platform_name', '_pick_evenly_spaced', 'select_same_burst_stack', '_candidate_safe_roots', 'safe_cache_index', '_safe_matches_row', 'find_cached_safe', '_candidate_archive_roots', 'archive_cache_index', 'find_cached_archive', '_cached_archive_dir', 'require_cdse_credentials', 'ensure_burst_archive', 'extract_archive', 'resolve_burst_product', 'cleanup_split_intermediate', 'split_orbit_output_path', 'run_split_orbit', 'gpt_for', 'require_gpt_product', 'require_gpt_success', 'remove_dimap_product', 'clean_product_paths', 'pair_marker_path', 'expected_stamps_export_files', 'missing_stamps_export_files', 'write_pair_marker', 'run_pair_export', 'existing_pair_export_result', 'existing_stack_export_results', 'stack_exports_ready', 'resolve_and_prepare', 'patch_input_summary', 'patch_inputs_ready', 'clean_stale_mt_prep_artifacts', 'pystamps_config_args', 'run_pystamps_command'):
    globals()[_helper_name] = getattr(nb_utils, _helper_name)
del _helper_name

if load_dotenv is not None:
    load_dotenv(REPO_ROOT / ".env")

PREFLIGHT_ONLY = os.getenv("SARPYX_PYSTAMPS_PREFLIGHT", "0") == "1"
RUN_BURST_SEARCH = os.getenv("SARPYX_PYSTAMPS_RUN_SEARCH", "0" if PREFLIGHT_ONLY else "1") == "1"
RUN_HEAVY_STEPS = (os.getenv("SARPYX_PYSTAMPS_RUN_HEAVY", "0") == "1") and not PREFLIGHT_ONLY
RUN_PYSTAMPS = RUN_HEAVY_STEPS and os.getenv("SARPYX_PYSTAMPS_RUN_PYSTAMPS", "1") == "1"
STREAM_STACK = os.getenv("SARPYX_PYSTAMPS_STREAM_STACK", "1") == "1"
CLEAN_INTERMEDIATES = os.getenv("SARPYX_PYSTAMPS_CLEAN_INTERMEDIATES", "1") == "1"

DATA_DIR = REPO_ROOT / "data"
BURST_DIR = DATA_DIR / "bursts"
OUTPUT_DIR = DATA_DIR / "output" / "sarpyx_pystamps_psi"
STATS_DIR = OUTPUT_DIR / "burst_search"
PROCESS_DIR = OUTPUT_DIR / "processing"
EXPORT_DIR = OUTPUT_DIR / "stamps_export"
TOOL_SHIM_DIR = OUTPUT_DIR / "tool_shims"
SNAP_USERDIR = Path(os.getenv("SNAP_USERDIR", REPO_ROOT / ".snap"))
SNAP_TMPDIR = Path(os.getenv("SNAP_TMPDIR", REPO_ROOT / ".tmp" / "sarpyx_pystamps_psi"))

if os.getenv("GPT_PATH"):
    GPT_PATH = Path(os.getenv("GPT_PATH", ""))
else:
    GPT_PATH = Path(shutil.which("gpt") or "gpt")

SARPYX_ROOT = Path(os.getenv("SARPYX_ROOT", REPO_ROOT.parent / "sarpyx"))
SNAP2STAMPS_ROOT = Path(os.getenv("SNAP2STAMPS_ROOT", REPO_ROOT.parent / "snap2stamps"))
STAMPS_ROOT_DEFAULT = REPO_ROOT.parent / "StaMPS"
STAMPS_ROOT = Path(os.getenv("STAMPS_ROOT", STAMPS_ROOT_DEFAULT if STAMPS_ROOT_DEFAULT.exists() else "")) if os.getenv("STAMPS_ROOT") or STAMPS_ROOT_DEFAULT.exists() else None


nb_utils.install_notebook_globals(globals())
CDSE_ENV_FILE = load_cdse_env_file()


ensure_runtime_dirs()
CDSE_USERNAME = first_env("CDSE_USERNAME", "CDSE_USR")
CDSE_PASSWORD = first_env("CDSE_PASSWORD", "CDSE_PSW")
nb_utils.install_notebook_globals(globals())

print(json.dumps({
    "repo_root": display_path(REPO_ROOT),
    "preflight_only": PREFLIGHT_ONLY,
    "run_burst_search": RUN_BURST_SEARCH,
    "run_heavy_steps": RUN_HEAVY_STEPS,
    "run_pystamps": RUN_PYSTAMPS,
    "stream_stack": STREAM_STACK,
    "clean_intermediates": CLEAN_INTERMEDIATES,
    "output_dir": display_path(OUTPUT_DIR),
    "tool_shim_dir": display_path(TOOL_SHIM_DIR),
    "gpt_path": GPT_PATH.as_posix(),
    "cdse_env_file": display_path(CDSE_ENV_FILE) if CDSE_ENV_FILE else None,
    "has_cdse_username": bool(CDSE_USERNAME),
    "has_cdse_password": bool(CDSE_PASSWORD),
}, indent=2))

In [ ]:
nb_utils.install_notebook_globals(globals())
TRIANGLE_PATH = resolve_executable(os.getenv("TRIANGLE"), "triangle")
SNAPHU_PATH = resolve_executable(os.getenv("SNAPHU"), "snaphu")
GPT_OK, GPT_MESSAGE = gpt_looks_like_snap(GPT_PATH)
nb_utils.install_notebook_globals(globals())

prerequisites = pd.DataFrame([
    {"name": "ESA SNAP gpt", "path": GPT_PATH.as_posix(), "ok": GPT_OK, "required_for": "SNAP processing", "detail": GPT_MESSAGE},
    {"name": "CDSE username", "path": "CDSE_USERNAME/CDSE_USR", "ok": bool(CDSE_USERNAME), "required_for": "burst download", "detail": "configured" if CDSE_USERNAME else "missing"},
    {"name": "CDSE password", "path": "CDSE_PASSWORD/CDSE_PSW", "ok": bool(CDSE_PASSWORD), "required_for": "burst download", "detail": "configured" if CDSE_PASSWORD else "missing"},
    {"name": "native Rustamps prep", "path": "pystamps.prep.prepare_snap_mt_prep_inputs", "ok": True, "required_for": "StaMPS prep", "detail": "packaged"},
    {"name": "triangle", "path": display_path(TRIANGLE_PATH), "ok": TRIANGLE_PATH is not None, "required_for": "Rustamps stage 6/7", "detail": "configured" if TRIANGLE_PATH else "missing"},
    {"name": "snaphu", "path": display_path(SNAPHU_PATH), "ok": SNAPHU_PATH is not None, "required_for": "Rustamps stage 6", "detail": "configured" if SNAPHU_PATH else "missing"},
])
display(prerequisites)

required_without_cache = {"ESA SNAP gpt", "triangle", "snaphu"}
missing_required = prerequisites.loc[
    prerequisites["name"].isin(required_without_cache) & ~prerequisites["ok"],
    "name",
].tolist()
if RUN_HEAVY_STEPS and missing_required:
    raise RuntimeError("Full execution requires missing prerequisites: " + ", ".join(missing_required))
if RUN_HEAVY_STEPS and (not CDSE_USERNAME or not CDSE_PASSWORD):
    print("CDSE credentials are not configured; cached SAFE products will be reused first, then missing bursts will prompt or fail at download time.")

## Burst search configuration

The default stack targets one same-burst Sentinel-1A PSI stack. The live search performed during planning found 21 S1A acquisitions for `BurstId=249405`, `IW2`, ascending relative orbit `117` between 2024-08-09 and 2025-04-06.


In [ ]:
AOI_WKT = "POLYGON((12.4 41.8, 12.6 41.8, 12.6 42.0, 12.4 42.0, 12.4 41.8))"
POLARISATION = "VV"
MIN_STACK_SIZE = int(os.getenv("SARPYX_PYSTAMPS_MIN_STACK", "20"))
TARGET_STACK_SIZE = int(os.getenv("SARPYX_PYSTAMPS_TARGET_STACK", "21"))
REQUIRE_SINGLE_PLATFORM = os.getenv("SARPYX_PYSTAMPS_REQUIRE_S1A", "1") == "1"
TARGET_ORBIT_DIRECTION = os.getenv("SARPYX_PYSTAMPS_ORBIT_DIRECTION", "ASCENDING").upper()
TARGET_RELATIVE_ORBIT = int(os.getenv("SARPYX_PYSTAMPS_RELATIVE_ORBIT", "117"))
TARGET_SWATH = os.getenv("SARPYX_PYSTAMPS_SWATH", "IW2")
TARGET_BURST_ID = int(os.getenv("SARPYX_PYSTAMPS_BURST_ID", "249405"))

SEARCH_CONFIG = {
    "search": {
        "aoi_wkt": AOI_WKT,
        "start_date": os.getenv("SARPYX_PYSTAMPS_START_DATE", "2024-08-02T00:00:00"),
        "end_date": os.getenv("SARPYX_PYSTAMPS_END_DATE", "2025-04-15T00:00:00"),
        "polarisation": POLARISATION,
        "orbit_direction": TARGET_ORBIT_DIRECTION,
        "relative_orbit": TARGET_RELATIVE_ORBIT,
        "platforms": ["A"] if REQUIRE_SINGLE_PLATFORM else ["all"],
        "swath_identifier": TARGET_SWATH,
        "burst_id": TARGET_BURST_ID,
        "max_results": int(os.getenv("SARPYX_PYSTAMPS_MAX_RESULTS", "1000")),
    },
    "output_statistics": STATS_DIR.as_posix(),
    "save_results_csv": True,
}

display(pd.DataFrame([{
    "aoi": AOI_WKT,
    "start_date": SEARCH_CONFIG["search"]["start_date"],
    "end_date": SEARCH_CONFIG["search"]["end_date"],
    "polarisation": POLARISATION,
    "min_stack_size": MIN_STACK_SIZE,
    "target_stack_size": TARGET_STACK_SIZE,
    "platforms": ",".join(SEARCH_CONFIG["search"]["platforms"]),
    "burst_id": TARGET_BURST_ID,
    "swath": TARGET_SWATH,
    "relative_orbit": TARGET_RELATIVE_ORBIT,
    "orbit_direction": TARGET_ORBIT_DIRECTION,
}]))

In [ ]:
if RUN_BURST_SEARCH:
    ensure_runtime_dirs()
    from phidown import plot_product_footprints, run_burst_workflow

    BURST_SEARCH_RESULT = run_burst_workflow(SEARCH_CONFIG)
    BURST_DF = BURST_SEARCH_RESULT["df"]
    VALIDATION = BURST_SEARCH_RESULT["validation"]
    if not VALIDATION["ok"]:
        raise RuntimeError(f"Burst search validation failed: {VALIDATION['errors']}")
    print(json.dumps({
        "rows": int(len(BURST_DF)),
        "warnings": VALIDATION.get("warnings", []),
        "orbit_direction": (BURST_SEARCH_RESULT.get("orbit") or {}).get("orbit_direction"),
        "relative_orbit": (BURST_SEARCH_RESULT.get("orbit") or {}).get("relative_orbit"),
    }, indent=2, default=str))
    display(BURST_DF.head(10))
else:
    cached_search = STATS_DIR / "search_results.csv"
    if cached_search.exists():
        BURST_DF = pd.read_csv(cached_search)
        BURST_SEARCH_RESULT = {"df": BURST_DF, "validation": {"ok": True, "warnings": ["search loaded from cache"]}, "paths": {"search_results": cached_search.as_posix()}}
        print(f"Burst search skipped; loaded cached results from {display_path(cached_search)}.")
        display(BURST_DF.head(10))
    else:
        BURST_SEARCH_RESULT = {"df": pd.DataFrame(), "validation": {"ok": True, "warnings": ["search skipped"]}, "paths": {}}
        BURST_DF = BURST_SEARCH_RESULT["df"]
        print("Burst search skipped. Set SARPYX_PYSTAMPS_RUN_SEARCH=1 to run the live CDSE query.")

In [ ]:
STACK_GROUP_COLUMNS = [
    "BurstId",
    "SwathIdentifier",
    "RelativeOrbitNumber",
    "OrbitDirection",
    "PolarisationChannels",
]


nb_utils.install_notebook_globals(globals())

SELECTED_STACK = select_same_burst_stack(BURST_DF, min_stack_size=MIN_STACK_SIZE, target_stack_size=TARGET_STACK_SIZE)
if SELECTED_STACK.empty:
    MASTER_DATE = None
    INSAR_DATASET = EXPORT_DIR / "INSAR_unresolved"
    print("No stack selected because burst search was skipped or empty.")
else:
    MASTER_ROW = SELECTED_STACK.loc[SELECTED_STACK["role"].eq("master")].iloc[0]
    MASTER_DATE = str(MASTER_ROW["date_yyyymmdd"])
    INSAR_DATASET = EXPORT_DIR / f"INSAR_{MASTER_DATE}"
    summary_cols = ["stack_index", "role", "Id", "BurstId", "SwathIdentifier", "RelativeOrbitNumber", "OrbitDirection", "PolarisationChannels", "ParentProductName", "date_yyyymmdd", "coverage"]
    display(SELECTED_STACK[[column for column in summary_cols if column in SELECTED_STACK.columns]])
    print(json.dumps({
        "selected_count": int(len(SELECTED_STACK)),
        "burst_id": int(MASTER_ROW["BurstId"]),
        "swath": str(MASTER_ROW["SwathIdentifier"]),
        "relative_orbit": int(MASTER_ROW["RelativeOrbitNumber"]),
        "orbit_direction": str(MASTER_ROW["OrbitDirection"]),
        "master_date": MASTER_DATE,
        "insar_dataset": display_path(INSAR_DATASET),
    }, indent=2))

nb_utils.install_notebook_globals(globals())

## Download and extract the burst stack

Burst archives are cached by UUID under `data/bursts/archives/`. Extraction is also cached, so rerunning the notebook resumes from local products where possible.


In [ ]:
ACCESS_TOKEN = None
SAFE_CACHE_INDEX: list[Path] | None = None
ARCHIVE_CACHE_INDEX: list[Path] | None = None
nb_utils.install_notebook_globals(globals())


if RUN_HEAVY_STEPS and not SELECTED_STACK.empty and not STREAM_STACK:
    ensure_runtime_dirs()
    product_rows = []
    for _, row in SELECTED_STACK.iterrows():
        resolved = resolve_burst_product(row)
        product_rows.append({**row.to_dict(), **resolved})
    STACK_PRODUCTS = pd.DataFrame(product_rows)
    display(STACK_PRODUCTS[["stack_index", "role", "date_yyyymmdd", "source", "archive", "safe_product"]])
elif RUN_HEAVY_STEPS and not SELECTED_STACK.empty and STREAM_STACK:
    ensure_runtime_dirs()
    STACK_PRODUCTS = SELECTED_STACK.copy()
    STACK_PRODUCTS["archive"] = "<resolved during streaming>"
    STACK_PRODUCTS["safe_product"] = "<resolved during streaming>"
    STACK_PRODUCTS["source"] = "stream_pending"
    print("Streaming mode enabled: bursts will be resolved one at a time during pair export.")
    display(STACK_PRODUCTS[["stack_index", "role", "date_yyyymmdd", "source", "archive", "safe_product"]])
else:
    STACK_PRODUCTS = SELECTED_STACK.copy()
    if not STACK_PRODUCTS.empty:
        cache_rows = []
        for _, row in STACK_PRODUCTS.iterrows():
            cached_archive = find_cached_archive(row)
            cached_safe = find_cached_safe(row)
            cache_rows.append({
                "archive": (cached_archive or (_cached_archive_dir(str(row["Id"])) / "<archive>.zip")).as_posix(),
                "safe_product": display_path(cached_safe or (BURST_DIR / "extracted" / str(row["Id"]) / "<product>.SAFE")),
                "source": "cached_safe" if cached_safe else ("cached_archive" if cached_archive else "pending_download"),
            })
        cache_df = pd.DataFrame(cache_rows, index=STACK_PRODUCTS.index)
        STACK_PRODUCTS = pd.concat([STACK_PRODUCTS, cache_df], axis=1)
    print("Download/extract skipped. Set SARPYX_PYSTAMPS_RUN_HEAVY=1 after configuring prerequisites to run it.")
    display(STACK_PRODUCTS[[column for column in ["stack_index", "role", "date_yyyymmdd", "source", "archive", "safe_product"] if column in STACK_PRODUCTS.columns]])

nb_utils.install_notebook_globals(globals())

## Split and apply orbit

Each selected burst product is prepared independently with TOPSAR split and orbit correction before pairwise coregistration.


In [ ]:
DEM_NAME = os.getenv("SARPYX_PYSTAMPS_DEM_NAME", "SRTM 1Sec HGT")
GPT_MEMORY = os.getenv("SARPYX_PYSTAMPS_GPT_MEMORY") or None
GPT_PARALLELISM = int(os.getenv("SARPYX_PYSTAMPS_GPT_PARALLELISM", "14"))
GPT_TIMEOUT = int(os.getenv("SARPYX_PYSTAMPS_GPT_TIMEOUT", "7200"))
SUBSET_REGION = os.getenv("SARPYX_PYSTAMPS_SUBSET_REGION", "0,0,0,0")
SPLIT_WITH_AOI = os.getenv("SARPYX_PYSTAMPS_SPLIT_WITH_AOI", "0") == "1"
SPLIT_AOI_WKT = AOI_WKT if SPLIT_WITH_AOI else None
nb_utils.install_notebook_globals(globals())


if RUN_HEAVY_STEPS and not STACK_PRODUCTS.empty and not STREAM_STACK:
    prepared_rows = []
    for _, row in STACK_PRODUCTS.iterrows():
        prepared = run_split_orbit(row)
        prepared_rows.append({**row.to_dict(), "prepared_product": prepared.as_posix()})
    PREPARED_STACK = pd.DataFrame(prepared_rows)
    display(PREPARED_STACK[["stack_index", "role", "date_yyyymmdd", "prepared_product"]])
elif RUN_HEAVY_STEPS and not STACK_PRODUCTS.empty and STREAM_STACK:
    PREPARED_STACK = STACK_PRODUCTS.copy()
    PREPARED_STACK["prepared_product"] = PREPARED_STACK.apply(lambda row: split_orbit_output_path(row).as_posix(), axis=1)
    print("Streaming mode enabled: split/orbit products will be generated on demand during pair export.")
    display(PREPARED_STACK[["stack_index", "role", "date_yyyymmdd", "prepared_product"]])
else:
    PREPARED_STACK = STACK_PRODUCTS.copy()
    if not PREPARED_STACK.empty:
        PREPARED_STACK["prepared_product"] = PREPARED_STACK.apply(lambda row: split_orbit_output_path(row).as_posix(), axis=1)
    print("Split/orbit processing skipped.")
    display(PREPARED_STACK[[column for column in ["stack_index", "role", "date_yyyymmdd", "prepared_product"] if column in PREPARED_STACK.columns]])

nb_utils.install_notebook_globals(globals())

## Pair processing and StaMPS export

For each secondary, the notebook builds a coregistered master-secondary pair, generates the interferogram, writes elevation/lat/lon/topo bands during topographic phase removal, subsets both products to the AOI, and exports into the shared `INSAR_<master_date>` folder.


In [ ]:
nb_utils.install_notebook_globals(globals())
if RUN_HEAVY_STEPS and not PREPARED_STACK.empty and stack_exports_ready(PREPARED_STACK):
    EXPORT_RESULTS = existing_stack_export_results(PREPARED_STACK)
    print("All expected StaMPS export files are present; skipping SNAP pair processing.")
    display(EXPORT_RESULTS)
elif RUN_HEAVY_STEPS and not PREPARED_STACK.empty and STREAM_STACK:
    master_row = PREPARED_STACK.loc[PREPARED_STACK["role"].eq("master")].iloc[0].copy()
    master_row = resolve_and_prepare(master_row)
    master_prepared = Path(master_row["prepared_product"])
    export_rows = []
    for _, raw_row in PREPARED_STACK.loc[PREPARED_STACK["role"].ne("master")].iterrows():
        row = resolve_and_prepare(raw_row.copy())
        result = run_pair_export(master_prepared, row)
        export_rows.append(result)
        clean_product_paths([row.get("prepared_product")])
        display(pd.DataFrame([result]))
    EXPORT_RESULTS = pd.DataFrame(export_rows)
    display(EXPORT_RESULTS)
elif RUN_HEAVY_STEPS and not PREPARED_STACK.empty:
    master_prepared = Path(PREPARED_STACK.loc[PREPARED_STACK["role"].eq("master"), "prepared_product"].iloc[0])
    secondary_rows = PREPARED_STACK.loc[PREPARED_STACK["role"].ne("master")].copy()
    export_rows = []
    for _, row in secondary_rows.iterrows():
        result = run_pair_export(master_prepared, row)
        export_rows.append(result)
        display(pd.DataFrame([result]))
    EXPORT_RESULTS = pd.DataFrame(export_rows)
    display(EXPORT_RESULTS)
else:
    EXPORT_RESULTS = pd.DataFrame(columns=["pair_id", "status", "export_marker"])
    print("Pair processing and StaMPS export skipped.")

nb_utils.install_notebook_globals(globals())

## StaMPS prep bridge

SNAP export alone is not the Rustamps input boundary. The native Rustamps prep step converts the exported `INSAR_<master>` folder into patch-level candidate inputs without calling `mt_prep_snap` or `gawk`. The notebook validates the patch tree before handing control to Rustamps.


In [ ]:
nb_utils.install_notebook_globals(globals())
from pystamps.prep import prepare_snap_mt_prep_inputs


def run_native_mt_prep(dataset_root: Path):
    if MASTER_DATE is None:
        raise RuntimeError("Cannot run native mt_prep without a selected master date")
    nb_utils.clean_stale_mt_prep_artifacts(dataset_root)
    started = time.perf_counter()
    summary = prepare_snap_mt_prep_inputs(
        dataset_root,
        master_date=MASTER_DATE,
        amp_dispersion=float(os.getenv("SARPYX_PYSTAMPS_AMP_DISPERSION", "0.4")),
        range_patches=int(os.getenv("SARPYX_PYSTAMPS_PATCH_RANGE", "1")),
        azimuth_patches=int(os.getenv("SARPYX_PYSTAMPS_PATCH_AZIMUTH", "1")),
        range_overlap=int(os.getenv("SARPYX_PYSTAMPS_PATCH_RANGE_SIZE", "50")),
        azimuth_overlap=int(os.getenv("SARPYX_PYSTAMPS_PATCH_AZIMUTH_SIZE", "50")),
        force=True,
    )
    elapsed = time.perf_counter() - started
    print(json.dumps({
        "processor": "pystamps.prep.prepare_snap_mt_prep_inputs",
        "patch_count": summary.patch_count,
        "candidate_count": summary.candidate_count,
        "elapsed_sec": elapsed,
    }, indent=2))
    return summary


FORCE_MT_PREP = os.getenv("SARPYX_PYSTAMPS_FORCE_MT_PREP", "0") == "1"
MT_PREP_SUMMARY = None

if RUN_HEAVY_STEPS:
    if FORCE_MT_PREP or not nb_utils.patch_inputs_ready(INSAR_DATASET):
        if FORCE_MT_PREP:
            print("Forcing native Rustamps prep to rebuild PATCH_* inputs.")
        MT_PREP_SUMMARY = run_native_mt_prep(INSAR_DATASET)
    if not nb_utils.patch_inputs_ready(INSAR_DATASET):
        raise RuntimeError("Native StaMPS prep finished but required Rustamps patch inputs are missing")
    PATCH_INPUTS = nb_utils.patch_input_summary(INSAR_DATASET)
    display(PATCH_INPUTS)
else:
    PATCH_INPUTS = nb_utils.patch_input_summary(INSAR_DATASET)
    print(f"StaMPS prep skipped. Expected dataset root: {display_path(INSAR_DATASET)}")
    display(PATCH_INPUTS if not PATCH_INPUTS.empty else pd.DataFrame([{"status": "no prepared PATCH_* inputs detected yet"}]))

nb_utils.install_notebook_globals(globals())

## Rustamps execution

After `mt_prep_snap` creates the patch inputs, Rustamps can run the staged PSI chain. The generated runtime config passes explicit `triangle` and `snaphu` paths when available.


In [ ]:
nb_utils.install_notebook_globals(globals())
if RUN_PYSTAMPS:
    if not nb_utils.patch_inputs_ready(INSAR_DATASET):
        raise RuntimeError("Rustamps cannot run until mt_prep_snap has created PATCH_* pscands inputs")
    DESCRIBE_INPUTS = nb_utils.run_pystamps_command(["describe-inputs", "--stage", "1", "--dataset", INSAR_DATASET.as_posix(), "--patch", "PATCH_1"])
    STATUS_BEFORE = nb_utils.run_pystamps_command(["status", "--dataset", INSAR_DATASET.as_posix()])
    RUN_REPORT = nb_utils.run_pystamps_command(["run", "--dataset", INSAR_DATASET.as_posix(), "--start-step", "1", "--end-step", "8"])
    STATUS_AFTER = nb_utils.run_pystamps_command(["status", "--dataset", INSAR_DATASET.as_posix()])
    display(pd.DataFrame(RUN_REPORT["stdout"]))
    print(json.dumps({"status_after": STATUS_AFTER["stdout"]}, indent=2, default=str))
else:
    DESCRIBE_INPUTS = STATUS_BEFORE = RUN_REPORT = STATUS_AFTER = None
    print("Rustamps execution skipped. Set SARPYX_PYSTAMPS_RUN_HEAVY=1 after configuring all prerequisites to run it.")

nb_utils.install_notebook_globals(globals())

## Final diagnostics

This section reports the dataset stage state and, when the final products exist, loads velocity/stability diagnostics through the packaged Rustamps notebook helpers.


In [ ]:
from pystamps.io.dataset import discover_dataset, infer_merged_stage, infer_patch_stage

if INSAR_DATASET.exists():
    layout = discover_dataset(INSAR_DATASET)
    status_rows = [{"scope": "merged", "target": INSAR_DATASET.name, "stage": infer_merged_stage(INSAR_DATASET)}]
    status_rows.extend({"scope": "patch", "target": patch.name, "stage": infer_patch_stage(patch)} for patch in layout.patches)
    display(pd.DataFrame(status_rows))
else:
    print(f"Dataset does not exist yet: {display_path(INSAR_DATASET)}")

final_products = {
    "mean_v.mat": INSAR_DATASET / "mean_v.mat",
    "uw_space_time.mat": INSAR_DATASET / "uw_space_time.mat",
    "scla2.mat": INSAR_DATASET / "scla2.mat",
    "phuw2.mat": INSAR_DATASET / "phuw2.mat",
}
display(pd.DataFrame([{"product": name, "exists": path.exists(), "path": display_path(path)} for name, path in final_products.items()]))

if all(path.exists() for path in (INSAR_DATASET / "ps2.mat", INSAR_DATASET / "mean_v.mat")):
    from pystamps.notebooks import load_velocity_diagnostics

    diagnostics = load_velocity_diagnostics(INSAR_DATASET, apply_step8=(INSAR_DATASET / "uw_space_time.mat").exists())
    display(pd.DataFrame([{
        "points": int(len(diagnostics.velocity)),
        "velocity_source": diagnostics.velocity_source,
        "stability_source": diagnostics.stability_source,
        "velocity_mean": float(np.nanmean(diagnostics.velocity)),
        "velocity_p10": float(np.nanpercentile(diagnostics.velocity, 10)),
        "velocity_p90": float(np.nanpercentile(diagnostics.velocity, 90)),
    }]))
else:
    print("Final velocity diagnostics are not available until Rustamps writes ps2.mat and mean_v.mat.")

## End-to-end logic and plots

The final PSI products are created through four handoff boundaries:

1. **Burst stack**: one same-burst Sentinel-1A geometry is selected, so all dates share burst id, swath, relative orbit, pass direction, and polarization.
2. **SARpyx/SNAP to SNAP2StaMPS**: each master-secondary pair is split, orbit-corrected, coregistered, debursted, interferogrammed, topo-corrected, subset to the AOI, and exported with `StampsExport`.
3. **Native Rustamps prep**: Rustamps turns the SNAP2StaMPS export tree into `PATCH_*` candidate files (`pscands.1.ij`, `.ph`, `.ll`, `.da`, `.hgt`).
4. **Rustamps PSI**: stages 1-8 select, merge, unwrap, correct SCLA terms, and write final velocity/time-series products such as `mean_v.mat` and `uw_space_time.mat`.

The code below stays intentionally small: all plotting, geospatial map rendering, and output summarization live in `notebooks/utils.py`.


In [ ]:
from notebooks import utils as nb_utils

PLOT_CONTEXT = nb_utils.build_plot_context(
    INSAR_DATASET,
    OUTPUT_DIR,
    stack_products=SELECTED_STACK if 'SELECTED_STACK' in globals() else None,
    stats_dir=STATS_DIR,
    master_date=MASTER_DATE,
)

for old_plot in PLOT_CONTEXT.plot_dir.glob('*.png'):
    old_plot.unlink()

PLOT_MANIFEST = nb_utils.render_all_plots(PLOT_CONTEXT, aoi_wkt=AOI_WKT)

display(nb_utils.pipeline_summary(PLOT_CONTEXT))
display(nb_utils.snap2stamps_output_summary(PLOT_CONTEXT))
display(PLOT_MANIFEST)

### Saved plot gallery

This gallery keeps the current diagnostic plots and adds the requested map and SNAP2StaMPS plots. The static burst and velocity maps use cached OpenStreetMap tiles when network access is available, and the burst-location helper also writes an interactive HTML map beside the PNG outputs.

In [ ]:
nb_utils.display_plot_gallery(PLOT_MANIFEST)